# TRIAD Detox: 3-Seed Benchmark Pipeline

**Principles**: Core code (reward/generate/Post-Val/check Val) = IDENTICAL to Table 2 (no batching, full Val, check every 10 steps) — Evaluation = Memory optimization: Release training model from GPU before sequentially evaluating checkpoints — Outputs = Standardized outputs (3 versions FULL/Toxic/Nontoxic for Table 3/4; Learning curves logged per seed; Main summary table with per-seed metrics and Mean ± Std; Bootstrap confidence intervals provided in appendix; Behavioral metrics and perplexity computed on Full test set).

All training and evaluation cells verify existing disk checkpoints (resume-safe) — if a failure occurs, simply re-run that specific cell.

## Stage 1: Setup & Dependencies

In [1]:
!pip install -q "trl==0.11.4" transformers sentence-transformers accelerate pandas scipy matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os, time, math, random, logging, json as pyjson, re, glob
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('RL-Pipeline')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
logger.info(f'Device: {DEVICE}')
assert os.path.exists('/content/drive/MyDrive'), 'Google Drive is not mounted! Please run drive.mount() or set local paths.'

import trl
logger.info(f'trl version: {trl.__version__} (expected 0.11.4)')

DATA_DIR = '/content/drive/MyDrive/Data_Khoaluan/03_stratified_benchmark'
T5_SFT_DIR = '/content/drive/MyDrive/Data_Khoaluan/T5_base'
RL_OUTPUT_DIR = '/content/drive/MyDrive/Data_Khoaluan/ppo_config_f_v1'
RESULTS_DIR = os.path.join(RL_OUTPUT_DIR, 'final_results')
os.makedirs(RL_OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED_LIST = [42, 123, 777]
logger.info(f'DATA_DIR={DATA_DIR} | T5_SFT_DIR={T5_SFT_DIR} | RL_OUTPUT_DIR={RL_OUTPUT_DIR}')
logger.info(f'SEED_LIST: {SEED_LIST}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Stage 1b: Optional Checkpoint & Output Reset

**Run this cell only once if you wish to reset checkpoints and outputs** — if resuming mid-run, DO NOT run this cell as it will delete all progress.

In [2]:
import shutil

CONFIRM_DELETE = False  # Set to True and re-run this cell only if you REALLY want to delete everything

if CONFIRM_DELETE:
    for SEED in SEED_LIST:
        seed_dir = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
        if os.path.exists(seed_dir):
            shutil.rmtree(seed_dir)
            print(f'Deleted checkpoint for Seed {SEED}.')
    old_files = glob.glob(os.path.join(RESULTS_DIR, '*_seed*.csv'))
    old_files += glob.glob(os.path.join(RESULTS_DIR, 'baseline_per_sample.csv'))
    for f in old_files:
        os.remove(f)
    print(f'Deleted {len(old_files)} legacy CSV files.')
else:
    print('CONFIRM_DELETE=False — Skipping deletion. Set CONFIRM_DELETE = True to reset outputs.')

CONFIRM_DELETE=False — nothing is deleted. Set to True if you want to wipe everything and restart from scratch.


## Stage 2: Load Stratified Benchmark Dataset

In [3]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
val_df   = pd.read_csv(os.path.join(DATA_DIR, 'val.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

train_toxic = train_df.reset_index(drop=True)
val_toxic   = val_df.reset_index(drop=True)
test_toxic  = test_df.reset_index(drop=True)
val_texts_all = val_toxic['text'].tolist()   # FULL Val — NO subsampling

logger.info(f'Loaded dataset: train={len(train_toxic)}, val={len(val_toxic)}, test={len(test_toxic)}.')
print(train_toxic['band'].value_counts())

band
easy      1149
medium     754
hard       521
Name: count, dtype: int64


## Stage 3-9: Reward Functions (Standard Per-Sample Formulation)

In [4]:
from transformers import T5TokenizerFast, AutoTokenizer, AutoModelForSequenceClassification
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from sentence_transformers import SentenceTransformer
import torch.nn.functional as F

tokenizer = T5TokenizerFast.from_pretrained(T5_SFT_DIR)
TASK_PREFIX = 'detoxify: '
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128

def to_list(x):
    if torch.is_tensor(x): return x.detach().cpu().tolist()
    if isinstance(x, np.ndarray): return x.tolist()
    return list(x)

tox_tokenizer = AutoTokenizer.from_pretrained('unitary/unbiased-toxic-roberta')
tox_model = AutoModelForSequenceClassification.from_pretrained('unitary/unbiased-toxic-roberta').to(DEVICE)
tox_model.eval()
id2label = tox_model.config.id2label
TOX_IDX = next((int(i) for i, l in id2label.items() if str(l).lower() in {'toxic','toxicity'}), 0)

@torch.no_grad()
def compute_r_tox(texts):
    enc = tox_tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
    probs = torch.sigmoid(tox_model(**enc).logits.float())[:, TOX_IDX]
    return to_list(1.0 - probs)

sbert_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE)

def compute_r_sem(X_texts, Y_texts):
    eX = sbert_model.encode(X_texts, convert_to_tensor=True, device=DEVICE)
    eY = sbert_model.encode(Y_texts, convert_to_tensor=True, device=DEVICE)
    return to_list((F.cosine_similarity(eX, eY, dim=1) + 1.0) / 2.0)

gpt2_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
gpt2_model = GPT2LMHeadModel.from_pretrained('gpt2').to(DEVICE)
gpt2_model.eval()
MAX_LOSS_CLAMP = 20.0

@torch.no_grad()
def compute_perplexity(text):
    enc = gpt2_tokenizer(text, return_tensors='pt', truncation=True, max_length=256).to(DEVICE)
    if enc.input_ids.size(1) < 2:
        return 1.0
    loss = torch.clamp(gpt2_model(enc.input_ids, labels=enc.input_ids).loss, max=MAX_LOSS_CLAMP)
    return torch.exp(loss).item()

def compute_r_flu(texts):
    """Standard per-sample evaluation (unbatched)."""
    return [1.0 / (1.0 + math.log(max(compute_perplexity(t), 1.0 + 1e-6)) / 5.0) for t in texts]

def compute_r_len(X_texts, Y_texts):
    rewards = []
    for x, y in zip(X_texts, Y_texts):
        x_len, y_len = max(len(x.split()), 1), len(y.split())
        ratio = y_len / x_len
        if 0.7 <= ratio <= 1.3: r = 1.0
        elif ratio < 0.7: r = y_len / (0.7 * x_len)
        else: r = (1.3 * x_len) / max(y_len, 1)
        rewards.append(r)
    return rewards

POLITENESS_LEXICON = [
    'please', 'could you', 'would you', 'might you', 'can you', 'will you',
    'hi', 'hello', 'hey', 'greetings', 'howdy', 'good morning', 'good evening', 'good afternoon',
    'goodbye', 'bye', 'see you later', 'thank', 'grateful', 'gratitude',
    'sorry', 'woops', 'oops', 'whoops', 'apologize',
    'is okay', 'not worry', 'no big deal', 'not a big deal', 'no problem', 'no worries',
    'is fine', 'you are good', "it's fine", "it's okay", 'let me know', 'for you',
]
P_SIZE = len(POLITENESS_LEXICON)

def compute_r_pol(Y_texts):
    return [min(sum(1 for w in POLITENESS_LEXICON if w in y.lower()) / P_SIZE, 1.0) for y in Y_texts]

# --- R_cov: content-word coverage (recall), mask toxic tokens before comparison — directly integrated into reward ---
!pip install -q spacy "transformers==4.38.2" transformers-interpret
import spacy
import pickle
try:
    nlp_cov = spacy.load('en_core_web_sm')
except OSError:
    !python -m spacy download en_core_web_sm
    nlp_cov = spacy.load('en_core_web_sm')

from transformers_interpret import SequenceClassificationExplainer
cov_explainer = SequenceClassificationExplainer(tox_model, tox_tokenizer)  # reuse existing tox_model above

CONTENT_POS = {'NOUN', 'PROPN', 'VERB', 'ADJ', 'ADV'}  # Jurafsky & Martin — open class (content words), NO lemmatization
CW_MASKED_CACHE = {}  # content_words(X) with toxic words masked — computed once per input X, reused thereafter

def content_words(text):
    return {tok.text.lower() for tok in nlp_cov(text) if tok.pos_ in CONTENT_POS}

def toxic_words(text, threshold=0.3):
    attrs = cov_explainer(text, class_name=id2label[TOX_IDX])
    return {w.lower() for w, score in attrs if score > threshold}

def compute_r_cov(X_texts, Y_texts):
    rewards = []
    for x, y in zip(X_texts, Y_texts):
        if x not in CW_MASKED_CACHE:
            CW_MASKED_CACHE[x] = content_words(x) - toxic_words(x)
        cwx = CW_MASKED_CACHE[x]
        cwy = content_words(y)
        inter = len(cwx & cwy)
        rewards.append(inter / max(len(cwx), 1))   # recall
    return rewards

# Precompute cache for all train_toxic (X fixed, reused across steps) — run once, save pickle
CW_CACHE_PATH = os.path.join(RESULTS_DIR, 'cw_masked_cache.pkl')
if os.path.exists(CW_CACHE_PATH):
    with open(CW_CACHE_PATH, 'rb') as f:
        CW_MASKED_CACHE.update(pickle.load(f))
    logger.info(f'[R_cov cache] Already cached — {len(CW_MASKED_CACHE)} inputs X.')
else:
    for i, x in enumerate(train_toxic['text'].unique()):
        if x not in CW_MASKED_CACHE:
            CW_MASKED_CACHE[x] = content_words(x) - toxic_words(x)
        if (i + 1) % 200 == 0:
            logger.info(f'[R_cov cache] {i+1} inputs X processed.')
    with open(CW_CACHE_PATH, 'wb') as f:
        pickle.dump(CW_MASKED_CACHE, f)
    logger.info(f'[R_cov cache] Calculation complete, cached {len(CW_MASKED_CACHE)} inputs X.')

def compute_r_total(X_texts, Y_texts, weights):
    r_tox, r_sem, r_flu = compute_r_tox(Y_texts), compute_r_sem(X_texts, Y_texts), compute_r_flu(Y_texts)
    r_len, r_pol = compute_r_len(X_texts, Y_texts), compute_r_pol(Y_texts)
    w_cov = weights.get('cov', 0.0)                                    # default 0 -> identical to LEGACY behavior
    r_cov = compute_r_cov(X_texts, Y_texts) if w_cov > 0 else [0.0] * len(Y_texts)
    total = [weights['tox']*r_tox[i] + weights['sem']*r_sem[i] + weights['flu']*r_flu[i]
             + weights['len']*r_len[i] + weights['pol']*r_pol[i] + w_cov*r_cov[i] for i in range(len(Y_texts))]
    return total, {'r_tox':r_tox,'r_sem':r_sem,'r_flu':r_flu,'r_len':r_len,'r_pol':r_pol,'r_cov':r_cov}

logger.info('Reward functions (standard, non-batched), including R_cov, ready.')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.11.4 requires transformers>=4.40.0, but you have transformers 4.38.2 which is incompatible.
sentence-transformers 5.7.0 requires tokenizers>=0.19, but you have tokenizers 0.15.2 which is incompatible.
sentence-transformers 5.7.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.


## Stage 10: PPO Configuration

In [5]:
from trl import PPOTrainer, PPOConfig, AutoModelForSeq2SeqLMWithValueHead, create_reference_model

BETA_KL, CLIP_EPS, GAE_GAMMA, GAE_LAMBDA = 0.05, 0.2, 0.99, 0.95
LEARNING_RATE, PPO_BATCH_SIZE, PPO_MINI_BATCH_SIZE = 1.4e-5, 8, 4

ppo_config = PPOConfig(
    learning_rate=LEARNING_RATE, batch_size=PPO_BATCH_SIZE, mini_batch_size=PPO_MINI_BATCH_SIZE,
    ppo_epochs=4, init_kl_coef=BETA_KL, cliprange=CLIP_EPS, cliprange_value=CLIP_EPS,
    gamma=GAE_GAMMA, lam=GAE_LAMBDA,
)
logger.info('PPOConfig ready.')

/usr/local/lib/python3.13/dist-packages/trl/trainer/ppo_config.py:207: FutureWarning: `PPOConfig` is deprecated and will be removed in the future. Please use `PPOv2Config` with `PPOv2Trainer` instead.
  warnings.warn(


## Stage 11: Generation, EarlyStopper & Phase Runner with Diagnostics

`CHECK_EVERY_STEPS=10` + FULL Val (standard evaluation, no subsampling). `run_phase_with_diagnostics` logs `stats` (reward/KL/policy_loss) at each step to CSV.

In [6]:
EVAL_BATCH_SIZE = 16
CHECK_EVERY_STEPS = 10   # Evaluates every 10 steps
MAX_STEPS_PER_PHASE = 200
# No VAL_SUBSAMPLE_SIZE — evaluate_on_val uses FULL val_texts

def generate_batch_deterministic(model, texts, batch_size=EVAL_BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    results = []
    n_batches = math.ceil(len(texts) / batch_size)
    for bi in range(n_batches):
        batch_texts = texts[bi*batch_size:(bi+1)*batch_size]
        inputs = [TASK_PREFIX + t for t in batch_texts]
        try:
            enc = tokenizer(inputs, return_tensors='pt', padding=True, truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
            with torch.no_grad():
                out_ids = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
            results.extend([tokenizer.decode(o, skip_special_tokens=True) for o in out_ids])
        except RuntimeError as e:
            logger.error(f'Batch {bi} error: {e}. Falling back to sentence-by-sentence evaluation.')
            for t in batch_texts:
                try:
                    enc1 = tokenizer(TASK_PREFIX+t, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
                    with torch.no_grad():
                        o = model.generate(**enc1, max_new_tokens=max_new_tokens, do_sample=False, num_beams=4)
                    results.append(tokenizer.decode(o[0], skip_special_tokens=True))
                except Exception as e2:
                    logger.error(f'Failed single sentence: {e2}')
                    results.append(t)
    return results

def evaluate_on_val(model, val_texts, weights):
    """Use FULL val_texts (standard evaluation, no subsampling)."""
    Y_val = generate_batch_deterministic(model, val_texts)
    rewards, _ = compute_r_total(val_texts, Y_val, weights)
    return float(np.mean(rewards))

class EarlyStopper:
    def __init__(self, patience=3):
        self.patience, self.best_score, self.num_bad_checks, self.should_stop = patience, -float('inf'), 0, False
    def check(self, score, step):
        if score > self.best_score:
            self.best_score, self.num_bad_checks = score, 0
            logger.info(f'[EarlyStop] step={step} J_val={score:.4f} -> IMPROVEMENT.')
            return True
        self.num_bad_checks += 1
        if self.num_bad_checks >= self.patience:
            self.should_stop = True
            logger.warning('[EarlyStop] Early stopping: patience exhausted — STOP.')
        return False

def run_phase_with_diagnostics(phase_name, phase_train_df, weights, val_texts,
                                  policy_model, ppo_trainer, save_dir_base, patience=3):
    """Log stats (reward/KL/policy_loss) at each step to CSV for diagnostics."""
    logger.info(f'STARTING {phase_name} — {len(phase_train_df)} rows, weights={weights}')
    stopper = EarlyStopper(patience=patience)
    texts_all = phase_train_df['text'].tolist()
    step = 0
    t_start = time.time()
    diagnostics = []

    while step < MAX_STEPS_PER_PHASE and not stopper.should_stop:
        idx = random.sample(range(len(texts_all)), min(PPO_BATCH_SIZE, len(texts_all)))
        X_batch = [texts_all[i] for i in idx]
        query_tensors = [tokenizer(TASK_PREFIX+x, return_tensors='pt', truncation=True,
                                     max_length=MAX_SOURCE_LEN).input_ids[0].to(DEVICE) for x in X_batch]
        response_tensors = ppo_trainer.generate(query_tensors, max_new_tokens=MAX_NEW_TOKENS,
                                                   do_sample=True, top_k=0, top_p=1.0, pad_token_id=tokenizer.pad_token_id)
        Y_batch = [tokenizer.decode(r, skip_special_tokens=True) for r in response_tensors]
        rewards_list, _ = compute_r_total(X_batch, Y_batch, weights)
        reward_tensors = [torch.tensor(r, dtype=torch.float32) for r in rewards_list]

        stats = ppo_trainer.step(query_tensors, response_tensors, reward_tensors)
        step += 1

        diagnostics.append({
            'step': step,
            'mean_reward': float(np.mean(rewards_list)),
            'kl': stats.get('objective/kl'),
            'policy_loss': stats.get('ppo/loss/policy'),
        })

        if step % 5 == 0:
            logger.info(f'[{phase_name}] step={step} R={np.mean(rewards_list):.4f} '
                        f'KL={diagnostics[-1]["kl"]} policy_loss={diagnostics[-1]["policy_loss"]}')

        if step % CHECK_EVERY_STEPS == 0:
            j_val = evaluate_on_val(policy_model, val_texts, weights)
            if stopper.check(j_val, step):
                save_dir = os.path.join(save_dir_base, phase_name.lower().replace(' ', '_'))
                policy_model.save_pretrained(save_dir)
                tokenizer.save_pretrained(save_dir)

    elapsed = time.time() - t_start
    logger.info(f'{phase_name} COMPLETED after {step} steps, {elapsed/60:.1f} mins. Best={stopper.best_score:.4f}')

    diag_df = pd.DataFrame(diagnostics)
    diag_path = os.path.join(save_dir_base, f'diagnostics_{phase_name.lower()}.csv')
    diag_df.to_csv(diag_path, index=False)
    logger.info(f'Saved training curves: {diag_path}')
    return stopper.best_score

logger.info('generate/EarlyStopper/run_phase_with_diagnostics ready.')

## Stage 17-18: Post-Validation & Checkpoint Evaluation

In [7]:
TAU_REDETECT, SEM_THRESHOLD, FLU_THRESHOLD, MAX_REGEN_ATTEMPTS = 0.30, 0.70, 0.70, 3
TEMPERATURES = [0.7, 0.9, 1.1]

def generate_one_with_temp(model, x_text, temperature):
    inputs = tokenizer(TASK_PREFIX+x_text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LEN).to(DEVICE)
    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
                                   temperature=temperature, top_k=0, top_p=1.0)
    return tokenizer.decode(out_ids[0], skip_special_tokens=True)

def passes_post_validation(x_text, y_text):
    t_out = 1.0 - compute_r_tox([y_text])[0]
    r_sem = compute_r_sem([x_text], [y_text])[0]
    r_flu = compute_r_flu([y_text])[0]
    y_len, x_len = len(y_text.split()), max(len(x_text.split()), 1)
    ratio = y_len / x_len
    return (t_out < TAU_REDETECT) and (r_sem > SEM_THRESHOLD) and (r_flu > FLU_THRESHOLD) and (0.7 <= ratio <= 1.3)

def generate_with_post_validation(model, X_texts):
    """Standard per-sample evaluation (unbatched)."""
    Y_out, n_failed = [], 0
    for x in X_texts:
        passed, last_y = False, None
        for attempt in range(MAX_REGEN_ATTEMPTS):
            y = generate_one_with_temp(model, x, TEMPERATURES[attempt])
            last_y = y
            if passes_post_validation(x, y):
                passed = True
                break
        if not passed:
            n_failed += 1
        Y_out.append(last_y)
    return Y_out, n_failed

def evaluate_checkpoint(model, eval_df, use_post_validation=False):
    X_texts, labels, bands = eval_df['text'].tolist(), eval_df['label'].tolist(), eval_df['band'].tolist()
    if use_post_validation:
        Y_texts, n_failed = generate_with_post_validation(model, X_texts)
    else:
        Y_texts, n_failed = generate_batch_deterministic(model, X_texts), None
    STA = compute_r_tox(Y_texts)
    SIM = compute_r_sem(X_texts, Y_texts)
    FLUE = compute_r_flu(Y_texts)
    Joint = [(s*si*f)**(1/3) for s,si,f in zip(STA,SIM,FLUE)]
    df = pd.DataFrame({'X':X_texts,'Y':Y_texts,'label':labels,'band':bands,'STA':STA,'Sim':SIM,'Flu':FLUE,'Joint':Joint})
    return df, n_failed

def summarize(df):
    return {'STA':float(df['STA'].mean()),'Sim':float(df['Sim'].mean()),'Flu':float(df['Flu'].mean()),'Joint':float(df['Joint'].mean())}

logger.info('Post-Validation (standard) + evaluate_checkpoint/summarize ready.')

## Baseline (SFT Model Evaluation): Full, Toxic, and Non-toxic Subsets

In [8]:
BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')

if os.path.exists(BASELINE_CSV):
    logger.info(f'Baseline already exists — skipping.')
    baseline_per_sample = pd.read_csv(BASELINE_CSV)
else:
    baseline_model = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(T5_SFT_DIR).to(DEVICE)
    baseline_model.eval()
    baseline_per_sample, _ = evaluate_checkpoint(baseline_model, test_toxic, use_post_validation=False)
    baseline_per_sample.to_csv(BASELINE_CSV, index=False)
    del baseline_model
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None

baseline_metrics_full = summarize(baseline_per_sample)
baseline_metrics_toxic = summarize(baseline_per_sample[baseline_per_sample['label']=='toxic'])
baseline_metrics_nontoxic = summarize(baseline_per_sample[baseline_per_sample['label']=='nontoxic'])
print(f'FULL Joint={baseline_metrics_full["Joint"]:.3f} | TOXIC Joint={baseline_metrics_toxic["Joint"]:.3f} | '
      f'NONTOXIC Joint={baseline_metrics_nontoxic["Joint"]:.3f}')

FULL Joint=0.759 | TOXIC Joint=0.758 | NONTOXIC Joint=0.764


## Seed 42: Training Curriculum

### [Seed 42] Phase 1 Training (Easy Difficulty) - Resume-Safe

In [9]:
SEED = 42
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase1_easy')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 1 (Easy) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = T5_SFT_DIR
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.45, 'sem': 0.2, 'flu': 0.1, 'len': 0.05, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'] == 'easy']).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE1_EASY', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 1 (Easy) train xong.')

### [Seed 42] Phase 2 Training (Easy + Medium Difficulty) - Resume-Safe

In [10]:
SEED = 42
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase2_balance')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 2 (+Med) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_42', 'phase1_easy')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.23, 'sem': 0.3, 'flu': 0.2, 'len': 0.07, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'].isin(['easy', 'medium'])]).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE2_BALANCE', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 2 (+Med) train xong.')

### [Seed 42] Phase 3 Training (Full Difficulty Curriculum) - Resume-Safe

In [11]:
SEED = 42
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 3 (Full) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_42', 'phase2_balance')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.15, 'sem': 0.3, 'flu': 0.2, 'len': 0.08, 'pol': 0.1, 'cov': 0.17}
    _phase_df = (train_toxic).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE3_REFINEMENT', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=5)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 3 (Full) train xong.')

## Seed 123: Training Curriculum

### [Seed 123] Phase 1 Training (Easy Difficulty) - Resume-Safe

In [12]:
SEED = 123
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase1_easy')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 1 (Easy) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = T5_SFT_DIR
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.45, 'sem': 0.2, 'flu': 0.1, 'len': 0.05, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'] == 'easy']).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE1_EASY', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 1 (Easy) train xong.')

### [Seed 123] Phase 2 Training (Easy + Medium Difficulty) - Resume-Safe

In [13]:
SEED = 123
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase2_balance')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 2 (+Med) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_123', 'phase1_easy')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.23, 'sem': 0.3, 'flu': 0.2, 'len': 0.07, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'].isin(['easy', 'medium'])]).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE2_BALANCE', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 2 (+Med) train xong.')

### [Seed 123] Phase 3 Training (Full Difficulty Curriculum) - Resume-Safe

In [14]:
SEED = 123
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 3 (Full) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_123', 'phase2_balance')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.15, 'sem': 0.3, 'flu': 0.2, 'len': 0.08, 'pol': 0.1, 'cov': 0.17}
    _phase_df = (train_toxic).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE3_REFINEMENT', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=5)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 3 (Full) train xong.')

## Seed 777: Training Curriculum

### [Seed 777] Phase 1 Training (Easy Difficulty) - Resume-Safe

In [15]:
SEED = 777
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase1_easy')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 1 (Easy) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = T5_SFT_DIR
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.45, 'sem': 0.2, 'flu': 0.1, 'len': 0.05, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'] == 'easy']).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE1_EASY', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 1 (Easy) train xong.')

ImportError: cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/usr/local/lib/python3.13/dist-packages/transformers/pytorch_utils.py)

### [Seed 777] Phase 2 Training (Easy + Medium Difficulty) - Resume-Safe

In [ ]:
SEED = 777
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase2_balance')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 2 (+Med) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_777', 'phase1_easy')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.23, 'sem': 0.3, 'flu': 0.2, 'len': 0.07, 'pol': 0.05, 'cov': 0.15}
    _phase_df = (train_toxic[train_toxic['band'].isin(['easy', 'medium'])]).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE2_BALANCE', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=3)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 2 (+Med) train xong.')

### [Seed 777] Phase 3 Training (Full Difficulty Curriculum) - Resume-Safe

In [ ]:
SEED = 777
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
os.makedirs(SEED_OUTPUT_DIR, exist_ok=True)
_ckpt_out = os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement')

if os.path.exists(_ckpt_out):
    logger.info(f'[Seed {SEED}] Phase 3 (Full) checkpoint exists — skipping.')
else:
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    _input_dir = os.path.join(RL_OUTPUT_DIR, 'seed_777', 'phase2_balance')
    _policy = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(_input_dir).to(DEVICE)
    _ref = create_reference_model(_policy).to(DEVICE)
    _trainer = PPOTrainer(config=ppo_config, model=_policy, ref_model=_ref, tokenizer=tokenizer)

    _WEIGHTS = {'tox': 0.15, 'sem': 0.3, 'flu': 0.2, 'len': 0.08, 'pol': 0.1, 'cov': 0.17}
    _phase_df = (train_toxic).reset_index(drop=True)
    run_phase_with_diagnostics('PHASE3_REFINEMENT', _phase_df, _WEIGHTS, val_texts_all, _policy, _trainer,
                                 SEED_OUTPUT_DIR, patience=5)

    del _policy, _ref, _trainer
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info('[Seed {SEED}] Phase 3 (Full) train xong.')

### [Seed 42] Learning Curves: Reward, KL Divergence, and Policy Loss

In [ ]:
SEED = 42
seed_dir = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
phase_diag_files = ['diagnostics_phase1_easy.csv', 'diagnostics_phase2_balance.csv', 'diagnostics_phase3_refinement.csv']

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle(f'Seed {SEED} — Training Curves')
for col, fname in enumerate(phase_diag_files):
    path = os.path.join(seed_dir, fname)
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    axes[0, col].plot(df['step'], df['mean_reward']); axes[0, col].set_title(f'{fname}\nReward')
    axes[1, col].plot(df['step'], df['kl']); axes[1, col].set_title('KL')
    axes[2, col].plot(df['step'], df['policy_loss']); axes[2, col].set_title('Policy Loss')
plt.tight_layout()
plt.savefig(f'/content/seed{SEED}_diagnostics.png')
plt.show()
print('Diagnostics: KL spike / reward drop mid-run -> instability (lower LR / increase BETA_KL).')
print('           All 3 curves smooth without large gains -> genuine variance, preserved as is.')

### [Seed 123] Learning Curves: Reward, KL Divergence, and Policy Loss

In [ ]:
SEED = 123
seed_dir = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
phase_diag_files = ['diagnostics_phase1_easy.csv', 'diagnostics_phase2_balance.csv', 'diagnostics_phase3_refinement.csv']

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle(f'Seed {SEED} — Training Curves')
for col, fname in enumerate(phase_diag_files):
    path = os.path.join(seed_dir, fname)
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    axes[0, col].plot(df['step'], df['mean_reward']); axes[0, col].set_title(f'{fname}\nReward')
    axes[1, col].plot(df['step'], df['kl']); axes[1, col].set_title('KL')
    axes[2, col].plot(df['step'], df['policy_loss']); axes[2, col].set_title('Policy Loss')
plt.tight_layout()
plt.savefig(f'/content/seed{SEED}_diagnostics.png')
plt.show()
print('Diagnostics: KL spike / reward drop mid-run -> instability (lower LR / increase BETA_KL).')
print('           All 3 curves smooth without large gains -> genuine variance, preserved as is.')

### [Seed 777] Learning Curves: Reward, KL Divergence, and Policy Loss

In [ ]:
SEED = 777
seed_dir = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')
phase_diag_files = ['diagnostics_phase1_easy.csv', 'diagnostics_phase2_balance.csv', 'diagnostics_phase3_refinement.csv']

fig, axes = plt.subplots(3, 3, figsize=(15, 9))
fig.suptitle(f'Seed {SEED} — Training Curves')
for col, fname in enumerate(phase_diag_files):
    path = os.path.join(seed_dir, fname)
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path)
    axes[0, col].plot(df['step'], df['mean_reward']); axes[0, col].set_title(f'{fname}\nReward')
    axes[1, col].plot(df['step'], df['kl']); axes[1, col].set_title('KL')
    axes[2, col].plot(df['step'], df['policy_loss']); axes[2, col].set_title('Policy Loss')
plt.tight_layout()
plt.savefig(f'/content/seed{SEED}_diagnostics.png')
plt.show()
print('Diagnostics: KL spike / reward drop mid-run -> instability (lower LR / increase BETA_KL).')
print('           All 3 curves smooth without large gains -> genuine variance, preserved as is.')

### [Seed 42] Memory-Optimized Sequential Checkpoint Evaluation

In [ ]:
SEED = 42
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')

# Release TRAIN model first (if still resident in RAM from training cell above)
for _v in ['_policy', '_ref', '_trainer']:
    if _v in dir():
        del globals()[_v]
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

CHECKPOINT_PATHS = {
    'Baseline (SFT)': T5_SFT_DIR,
    'Phase 1 (Easy)': os.path.join(SEED_OUTPUT_DIR, 'phase1_easy'),
    'Phase 2 (+Med)': os.path.join(SEED_OUTPUT_DIR, 'phase2_balance'),
    'Phase 3 (Full)': os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement'),
}

for name, path in CHECKPOINT_PATHS.items():
    if name == 'Baseline (SFT)':
        continue  # pre-calculated in Baseline cell
    if not os.path.exists(path):
        logger.warning(f'Missing checkpoint {name} — run corresponding training cell first.')
        continue
    safe = os.path.basename(path)
    out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_{safe}.csv')
    if os.path.exists(out_csv):
        logger.info(f'[Seed {SEED}] {name} already evaluated — skipping.')
        continue
    m = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(path).to(DEVICE)
    m.eval()
    per_sample_df, _ = evaluate_checkpoint(m, test_toxic, use_post_validation=False)
    per_sample_df.to_csv(out_csv, index=False)
    del m
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Evaluation of {name} completed.')

# Phase 3 + Post-Val — separate
pv_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase3_postval.csv')
p3_path = CHECKPOINT_PATHS['Phase 3 (Full)']
if os.path.exists(pv_csv):
    logger.info(f'[Seed {SEED}] Phase3+Post-Val already exists — skipping.')
elif os.path.exists(p3_path):
    m_p3 = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(p3_path).to(DEVICE)
    m_p3.eval()
    per_sample_pv, n_failed_pv = evaluate_checkpoint(m_p3, test_toxic, use_post_validation=True)
    per_sample_pv.to_csv(pv_csv, index=False)
    del m_p3
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Phase3+Post-Val finished: {n_failed_pv}/{len(test_toxic)} failed all 3 attempts.')

### [Seed 123] Memory-Optimized Sequential Checkpoint Evaluation

In [ ]:
SEED = 123
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')

# Release TRAIN model first (if still resident in RAM from training cell above)
for _v in ['_policy', '_ref', '_trainer']:
    if _v in dir():
        del globals()[_v]
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

CHECKPOINT_PATHS = {
    'Baseline (SFT)': T5_SFT_DIR,
    'Phase 1 (Easy)': os.path.join(SEED_OUTPUT_DIR, 'phase1_easy'),
    'Phase 2 (+Med)': os.path.join(SEED_OUTPUT_DIR, 'phase2_balance'),
    'Phase 3 (Full)': os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement'),
}

for name, path in CHECKPOINT_PATHS.items():
    if name == 'Baseline (SFT)':
        continue  # pre-calculated in Baseline cell
    if not os.path.exists(path):
        logger.warning(f'Missing checkpoint {name} — run corresponding training cell first.')
        continue
    safe = os.path.basename(path)
    out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_{safe}.csv')
    if os.path.exists(out_csv):
        logger.info(f'[Seed {SEED}] {name} already evaluated — skipping.')
        continue
    m = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(path).to(DEVICE)
    m.eval()
    per_sample_df, _ = evaluate_checkpoint(m, test_toxic, use_post_validation=False)
    per_sample_df.to_csv(out_csv, index=False)
    del m
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Evaluation of {name} completed.')

# Phase 3 + Post-Val — separate
pv_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase3_postval.csv')
p3_path = CHECKPOINT_PATHS['Phase 3 (Full)']
if os.path.exists(pv_csv):
    logger.info(f'[Seed {SEED}] Phase3+Post-Val already exists — skipping.')
elif os.path.exists(p3_path):
    m_p3 = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(p3_path).to(DEVICE)
    m_p3.eval()
    per_sample_pv, n_failed_pv = evaluate_checkpoint(m_p3, test_toxic, use_post_validation=True)
    per_sample_pv.to_csv(pv_csv, index=False)
    del m_p3
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Phase3+Post-Val finished: {n_failed_pv}/{len(test_toxic)} failed all 3 attempts.')

### [Seed 777] Memory-Optimized Sequential Checkpoint Evaluation

In [ ]:
SEED = 777
SEED_OUTPUT_DIR = os.path.join(RL_OUTPUT_DIR, f'seed_{SEED}')

# Release TRAIN model first (if still resident in RAM from training cell above)
for _v in ['_policy', '_ref', '_trainer']:
    if _v in dir():
        del globals()[_v]
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

CHECKPOINT_PATHS = {
    'Baseline (SFT)': T5_SFT_DIR,
    'Phase 1 (Easy)': os.path.join(SEED_OUTPUT_DIR, 'phase1_easy'),
    'Phase 2 (+Med)': os.path.join(SEED_OUTPUT_DIR, 'phase2_balance'),
    'Phase 3 (Full)': os.path.join(SEED_OUTPUT_DIR, 'phase3_refinement'),
}

for name, path in CHECKPOINT_PATHS.items():
    if name == 'Baseline (SFT)':
        continue  # pre-calculated in Baseline cell
    if not os.path.exists(path):
        logger.warning(f'Missing checkpoint {name} — run corresponding training cell first.')
        continue
    safe = os.path.basename(path)
    out_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_{safe}.csv')
    if os.path.exists(out_csv):
        logger.info(f'[Seed {SEED}] {name} already evaluated — skipping.')
        continue
    m = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(path).to(DEVICE)
    m.eval()
    per_sample_df, _ = evaluate_checkpoint(m, test_toxic, use_post_validation=False)
    per_sample_df.to_csv(out_csv, index=False)
    del m
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Evaluation of {name} completed.')

# Phase 3 + Post-Val — separate
pv_csv = os.path.join(RESULTS_DIR, f'per_sample_seed{SEED}_phase3_postval.csv')
p3_path = CHECKPOINT_PATHS['Phase 3 (Full)']
if os.path.exists(pv_csv):
    logger.info(f'[Seed {SEED}] Phase3+Post-Val already exists — skipping.')
elif os.path.exists(p3_path):
    m_p3 = AutoModelForSeq2SeqLMWithValueHead.from_pretrained(p3_path).to(DEVICE)
    m_p3.eval()
    per_sample_pv, n_failed_pv = evaluate_checkpoint(m_p3, test_toxic, use_post_validation=True)
    per_sample_pv.to_csv(pv_csv, index=False)
    del m_p3
    torch.cuda.empty_cache() if DEVICE == 'cuda' else None
    logger.info(f'[Seed {SEED}] Phase3+Post-Val finished: {n_failed_pv}/{len(test_toxic)} failed all 3 attempts.')

### [Seed 42] Metric Aggregation: Full, Toxic, and Non-toxic Subsets

In [ ]:
SEED = 42
phase_files = {
    'Phase 1 (Easy)': f'per_sample_seed{SEED}_phase1_easy.csv',
    'Phase 2 (+Med)': f'per_sample_seed{SEED}_phase2_balance.csv',
    'Phase 3 (Full)': f'per_sample_seed{SEED}_phase3_refinement.csv',
    'Phase 3+Post-Val.': f'per_sample_seed{SEED}_phase3_postval.csv',
}

t3, t3b, t3c = [{'Phase':'Baseline (SFT)', **baseline_metrics_full}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_toxic}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_nontoxic}]

phase3_df_this_seed = None
for name, fname in phase_files.items():
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(fpath):
        logger.warning(f'[Seed {SEED}] Missing {fpath}.')
        continue
    df = pd.read_csv(fpath)
    if name == 'Phase 3 (Full)':
        phase3_df_this_seed = df
    t3.append({'Phase': name, **summarize(df)})
    t3b.append({'Phase': name, **summarize(df[df['label']=='toxic'])})
    t3c.append({'Phase': name, **summarize(df[df['label']=='nontoxic'])})

pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

print(f'=== SEED {SEED} — TABLE 3 (FULL) ===')
print(pd.DataFrame(t3).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3b (Toxic) ===')
print(pd.DataFrame(t3b).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3c (Non-toxic) ===')
print(pd.DataFrame(t3c).round(3).to_string(index=False))

# --- Table 4/4b/4c (theo band, Phase 3) ---
if phase3_df_this_seed is not None:
    t4, t4b, t4c = [], [], []
    for band in ['easy', 'medium', 'hard']:
        bf = phase3_df_this_seed[phase3_df_this_seed['band']==band]
        bt = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='toxic')]
        bn = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='nontoxic')]
        if len(bf)>0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize(bf)})
        if len(bt)>0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize(bt)})
        if len(bn)>0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize(bn)})
    t4.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed), **summarize(phase3_df_this_seed)})
    t4b.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic'])})
    t4c.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic'])})

    pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

    print(f'\n=== SEED {SEED} — TABLE 4 (FULL, band) ===')
    print(pd.DataFrame(t4).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4b (Toxic, band) ===')
    print(pd.DataFrame(t4b).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4c (Non-toxic, band) ===')
    print(pd.DataFrame(t4c).round(3).to_string(index=False))

### [Seed 123] Metric Aggregation: Full, Toxic, and Non-toxic Subsets

In [ ]:
SEED = 123
phase_files = {
    'Phase 1 (Easy)': f'per_sample_seed{SEED}_phase1_easy.csv',
    'Phase 2 (+Med)': f'per_sample_seed{SEED}_phase2_balance.csv',
    'Phase 3 (Full)': f'per_sample_seed{SEED}_phase3_refinement.csv',
    'Phase 3+Post-Val.': f'per_sample_seed{SEED}_phase3_postval.csv',
}

t3, t3b, t3c = [{'Phase':'Baseline (SFT)', **baseline_metrics_full}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_toxic}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_nontoxic}]

phase3_df_this_seed = None
for name, fname in phase_files.items():
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(fpath):
        logger.warning(f'[Seed {SEED}] Missing {fpath}.')
        continue
    df = pd.read_csv(fpath)
    if name == 'Phase 3 (Full)':
        phase3_df_this_seed = df
    t3.append({'Phase': name, **summarize(df)})
    t3b.append({'Phase': name, **summarize(df[df['label']=='toxic'])})
    t3c.append({'Phase': name, **summarize(df[df['label']=='nontoxic'])})

pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

print(f'=== SEED {SEED} — TABLE 3 (FULL) ===')
print(pd.DataFrame(t3).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3b (Toxic) ===')
print(pd.DataFrame(t3b).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3c (Non-toxic) ===')
print(pd.DataFrame(t3c).round(3).to_string(index=False))

# --- Table 4/4b/4c (theo band, Phase 3) ---
if phase3_df_this_seed is not None:
    t4, t4b, t4c = [], [], []
    for band in ['easy', 'medium', 'hard']:
        bf = phase3_df_this_seed[phase3_df_this_seed['band']==band]
        bt = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='toxic')]
        bn = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='nontoxic')]
        if len(bf)>0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize(bf)})
        if len(bt)>0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize(bt)})
        if len(bn)>0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize(bn)})
    t4.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed), **summarize(phase3_df_this_seed)})
    t4b.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic'])})
    t4c.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic'])})

    pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

    print(f'\n=== SEED {SEED} — TABLE 4 (FULL, band) ===')
    print(pd.DataFrame(t4).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4b (Toxic, band) ===')
    print(pd.DataFrame(t4b).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4c (Non-toxic, band) ===')
    print(pd.DataFrame(t4c).round(3).to_string(index=False))

### [Seed 777] Metric Aggregation: Full, Toxic, and Non-toxic Subsets

In [ ]:
SEED = 777
phase_files = {
    'Phase 1 (Easy)': f'per_sample_seed{SEED}_phase1_easy.csv',
    'Phase 2 (+Med)': f'per_sample_seed{SEED}_phase2_balance.csv',
    'Phase 3 (Full)': f'per_sample_seed{SEED}_phase3_refinement.csv',
    'Phase 3+Post-Val.': f'per_sample_seed{SEED}_phase3_postval.csv',
}

t3, t3b, t3c = [{'Phase':'Baseline (SFT)', **baseline_metrics_full}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_toxic}], \
               [{'Phase':'Baseline (SFT)', **baseline_metrics_nontoxic}]

phase3_df_this_seed = None
for name, fname in phase_files.items():
    fpath = os.path.join(RESULTS_DIR, fname)
    if not os.path.exists(fpath):
        logger.warning(f'[Seed {SEED}] Missing {fpath}.')
        continue
    df = pd.read_csv(fpath)
    if name == 'Phase 3 (Full)':
        phase3_df_this_seed = df
    t3.append({'Phase': name, **summarize(df)})
    t3b.append({'Phase': name, **summarize(df[df['label']=='toxic'])})
    t3c.append({'Phase': name, **summarize(df[df['label']=='nontoxic'])})

pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

print(f'=== SEED {SEED} — TABLE 3 (FULL) ===')
print(pd.DataFrame(t3).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3b (Toxic) ===')
print(pd.DataFrame(t3b).round(3).to_string(index=False))
print(f'\n=== SEED {SEED} — TABLE 3c (Non-toxic) ===')
print(pd.DataFrame(t3c).round(3).to_string(index=False))

# --- Table 4/4b/4c (theo band, Phase 3) ---
if phase3_df_this_seed is not None:
    t4, t4b, t4c = [], [], []
    for band in ['easy', 'medium', 'hard']:
        bf = phase3_df_this_seed[phase3_df_this_seed['band']==band]
        bt = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='toxic')]
        bn = phase3_df_this_seed[(phase3_df_this_seed['band']==band)&(phase3_df_this_seed['label']=='nontoxic')]
        if len(bf)>0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize(bf)})
        if len(bt)>0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize(bt)})
        if len(bn)>0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize(bn)})
    t4.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed), **summarize(phase3_df_this_seed)})
    t4b.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='toxic'])})
    t4c.append({'Difficulty':'Overall','Count':len(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic']),
                 **summarize(phase3_df_this_seed[phase3_df_this_seed['label']=='nontoxic'])})

    pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

    print(f'\n=== SEED {SEED} — TABLE 4 (FULL, band) ===')
    print(pd.DataFrame(t4).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4b (Toxic, band) ===')
    print(pd.DataFrame(t4b).round(3).to_string(index=False))
    print(f'\n=== SEED {SEED} — TABLE 4c (Non-toxic, band) ===')
    print(pd.DataFrame(t4c).round(3).to_string(index=False))

## Main Summary Table: Per-Seed Results and Mean ± Std

**Official benchmark summary table for reporting** — reports all individual seeds, not just the mean.

In [ ]:
def print_main_table(prefix, index_col='Phase'):
    dfs = {}
    for SEED in SEED_LIST:
        path = os.path.join(RESULTS_DIR, f'{prefix}_seed{SEED}.csv')
        if os.path.exists(path):
            dfs[SEED] = pd.read_csv(path).set_index(index_col)
    if not dfs:
        print(f'No {prefix}_seed*.csv files found.')
        return
    keys = dfs[SEED_LIST[0]].index.tolist()
    for k in keys:
        print(f'--- {k} ---')
        for metric in ['STA', 'Sim', 'Flu', 'Joint']:
            values = []
            for SEED in SEED_LIST:
                if SEED in dfs and k in dfs[SEED].index:
                    v = dfs[SEED].loc[k, metric]
                    values.append(v)
            if not values:
                continue
            seed_str = ' | '.join(f'S{SEED}={v:.3f}' for SEED, v in zip(SEED_LIST, values))
            mean, std = np.mean(values), (np.std(values, ddof=1) if len(values)>1 else 0.0)
            print(f'  {metric:5s}: {seed_str}  ->  Mean±Std = {mean:.4f} ± {std:.4f}')
    print()

for prefix, idx_col in [('table3','Phase'), ('table3b','Phase'), ('table3c','Phase'),
                           ('table4','Difficulty'), ('table4b','Difficulty'), ('table4c','Difficulty')]:
    print('='*90)
    print(f'MAIN TABLE — {prefix}')
    print('='*90)
    print_main_table(prefix, idx_col)

## Appendix: Bootstrap Confidence Intervals (95% CI)

Inter-seed variance is reported via empirical Mean ± Std across seeds — using Bootstrap CI as primary would obscure true variance. The PRIMARY table remains the section above (individual seeds + Mean ± Std).

In [ ]:
def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values); n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)])
    lo = np.percentile(boot_means, (1-ci)/2*100)
    hi = np.percentile(boot_means, (1+ci)/2*100)
    return values.mean(), lo, hi

_all = []
for SEED in SEED_LIST:
    for fname, phase_name in [(f'per_sample_seed{SEED}_phase1_easy.csv','Phase 1 (Easy)'),
                                (f'per_sample_seed{SEED}_phase2_balance.csv','Phase 2 (+Med)'),
                                (f'per_sample_seed{SEED}_phase3_refinement.csv','Phase 3 (Full)'),
                                (f'per_sample_seed{SEED}_phase3_postval.csv','Phase 3+Post-Val.')]:
        fpath = os.path.join(RESULTS_DIR, fname)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df['seed'], df['phase'] = SEED, phase_name
            _all.append(df)
all_per_sample_pooled = pd.concat(_all, ignore_index=True)
print(f'Total per-sample lines reloaded: {len(all_per_sample_pooled)} rows')

def build_ci_table_pooled(df, group_col='phase', label_filter=None):
    sub = df if label_filter is None else df[df['label']==label_filter]
    rows = []
    for g in sub[group_col].unique():
        gdf = sub[sub[group_col]==g]
        row = {group_col: g, 'n_pooled': len(gdf)}
        for col in ['STA','Sim','Flu','Joint']:
            mean, lo, hi = bootstrap_ci(gdf[col].values)
            row[col] = round(mean,4); row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
        rows.append(row)
    return pd.DataFrame(rows)

for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NON-TOXIC', 'nontoxic')]:
    ci_table = build_ci_table_pooled(all_per_sample_pooled, label_filter=filt)
    print(f'\n=== Bootstrap CI (Appendix) — {label} ===')
    print(ci_table.to_string(index=False))
    ci_table.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower().replace("-","")}.csv'), index=False)

## Behavioral Metrics (Levenshtein Distance, Length Ratio) & Perplexity

In [ ]:
def tokenize_simple(text): return str(text).lower().split()

def levenshtein_token(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[n][m]

def compute_edit_behavior(row):
    x, y = str(row['X']).strip(), str(row['Y']).strip()
    xt, yt = tokenize_simple(x), tokenize_simple(y)
    xl = max(len(xt),1); yl = len(yt)
    return pd.Series({'unchanged': x.lower()==y.lower(), 'token_edit_rate': levenshtein_token(xt,yt)/xl,
                        'truncated': yl<xl, 'clause_loss': ((xl-yl)/xl) > 0.30})

print('=== TABLE OF 4 BEHAVIORAL METRICS — FULL ===')
results = []
for phase_name in all_per_sample_pooled['phase'].unique():
    sub = all_per_sample_pooled[all_per_sample_pooled['phase']==phase_name]
    m = sub.apply(compute_edit_behavior, axis=1)
    results.append({'Phase':phase_name,'n':len(sub),
                      'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                      'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                      'Short-fail Rate (%)':round(m['truncated'].mean()*100,1),
                      'Truncation >30% Rate (%)':round(m['clause_loss'].mean()*100,1)})
edit_behavior_full = pd.DataFrame(results)
print(edit_behavior_full.to_string(index=False))
edit_behavior_full.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_full.csv'), index=False)

print('\n=== TABLE OF PPL — FULL ===')
ppl_rows = []
for phase_name in all_per_sample_pooled['phase'].unique():
    pdata = all_per_sample_pooled[all_per_sample_pooled['phase']==phase_name]
    ppls = pd.Series([compute_perplexity(str(y)) for y in pdata['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows.append({'phase':phase_name,'mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f'{phase_name:20s}: mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}')
ppl_full = pd.DataFrame(ppl_rows)
ppl_full.to_csv(os.path.join(RESULTS_DIR, 'ppl_full.csv'), index=False)
logger.info('PIPELINE EXECUTION COMPLETED.')

In [ ]:
print('\n=== TABLE OF PPL — Toxic-only ===')
ppl_rows_toxic = []
_pooled_toxic = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
for phase_name in _pooled_toxic['phase'].unique():
    pdata = _pooled_toxic[_pooled_toxic['phase']==phase_name]
    ppls = pd.Series([compute_perplexity(str(y)) for y in pdata['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows_toxic.append({'phase':phase_name,'mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f'{phase_name:20s}: mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}')
ppl_toxic = pd.DataFrame(ppl_rows_toxic)
ppl_toxic.to_csv(os.path.join(RESULTS_DIR, 'ppl_toxic_only.csv'), index=False)

In [ ]:
def tokenize_simple(text): return str(text).lower().split()

def levenshtein_token(a, b):
    n, m = len(a), len(b)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(n+1): dp[i][0] = i
    for j in range(m+1): dp[0][j] = j
    for i in range(1, n+1):
        for j in range(1, m+1):
            dp[i][j] = dp[i-1][j-1] if a[i-1]==b[j-1] else 1+min(dp[i-1][j],dp[i][j-1],dp[i-1][j-1])
    return dp[n][m]

def compute_edit_behavior(row):
    x, y = str(row['X']).strip(), str(row['Y']).strip()
    xt, yt = tokenize_simple(x), tokenize_simple(y)
    xl = max(len(xt),1); yl = len(yt)
    return pd.Series({'unchanged': x.lower()==y.lower(), 'token_edit_rate': levenshtein_token(xt,yt)/xl,
                        'truncated': yl<xl, 'clause_loss': ((xl-yl)/xl) > 0.30})

print('=== TABLE OF 4 BEHAVIORAL METRICS — FULL ===')
results = []
for phase_name in all_per_sample_pooled['phase'].unique():
    sub = all_per_sample_pooled[all_per_sample_pooled['phase']==phase_name]
    m = sub.apply(compute_edit_behavior, axis=1)
    results.append({'Phase':phase_name,'n':len(sub),
                      'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                      'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                      'Short-fail Rate (%)':round(m['truncated'].mean()*100,1),
                      'Truncation >30% Rate (%)':round(m['clause_loss'].mean()*100,1)})
edit_behavior_full = pd.DataFrame(results)
print(edit_behavior_full.to_string(index=False))
edit_behavior_full.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_full.csv'), index=False)

print('\n=== TABLE OF 4 BEHAVIORAL METRICS — Toxic-only ===')
results_toxic = []
_pooled_toxic_behavior = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
for phase_name in _pooled_toxic_behavior['phase'].unique():
    sub = _pooled_toxic_behavior[_pooled_toxic_behavior['phase']==phase_name]
    m = sub.apply(compute_edit_behavior, axis=1)
    results_toxic.append({'Phase':phase_name,'n':len(sub),
                      'Unchanged Rate (%)':round(m['unchanged'].mean()*100,1),
                      'Token Edit Rate (%)':round(m['token_edit_rate'].mean()*100,1),
                      'Short-fail Rate (%)':round(m['truncated'].mean()*100,1),
                      'Truncation >30% Rate (%)':round(m['clause_loss'].mean()*100,1)})
edit_behavior_toxic = pd.DataFrame(results_toxic)
print(edit_behavior_toxic.to_string(index=False))
edit_behavior_toxic.to_csv(os.path.join(RESULTS_DIR, 'edit_behavior_toxic_only.csv'), index=False)

print('\n=== TABLE OF PPL — FULL ===')
ppl_rows = []
for phase_name in all_per_sample_pooled['phase'].unique():
    pdata = all_per_sample_pooled[all_per_sample_pooled['phase']==phase_name]
    ppls = pd.Series([compute_perplexity(str(y)) for y in pdata['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows.append({'phase':phase_name,'mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f'{phase_name:20s}: mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}')
ppl_full = pd.DataFrame(ppl_rows)
ppl_full.to_csv(os.path.join(RESULTS_DIR, 'ppl_full.csv'), index=False)

print('\n=== TABLE OF PPL — Toxic-only ===')
ppl_rows_toxic = []
_pooled_toxic = all_per_sample_pooled[all_per_sample_pooled['label']=='toxic']
for phase_name in _pooled_toxic['phase'].unique():
    pdata = _pooled_toxic[_pooled_toxic['phase']==phase_name]
    ppls = pd.Series([compute_perplexity(str(y)) for y in pdata['Y']])
    pct = ((ppls>=20)&(ppls<=150)).mean()
    ppl_rows_toxic.append({'phase':phase_name,'mean_PPL':round(ppls.mean(),1),'median_PPL':round(ppls.median(),1),'pct_in_20_150':round(pct,3)})
    print(f'{phase_name:20s}: mean={ppls.mean():7.1f}, trong[20,150]={pct:.1%}')
ppl_toxic = pd.DataFrame(ppl_rows_toxic)
ppl_toxic.to_csv(os.path.join(RESULTS_DIR, 'ppl_toxic_only.csv'), index=False)
logger.info('PIPELINE EXECUTION COMPLETED.')

In [ ]:
# =====================================================================================
# INDEPENDENT STA (fix N4) — FULL PIPELINE
# 1) Load s-nlp/roberta_toxicity_classifier, rescore STA on all per_sample_*.csv
# 2) Rebuild table3/3b/3c + table4/4b/4c (add STA (indep.) column)
# 3) Rebuild Bootstrap CI (FULL/TOXIC/NON-TOXIC) (add STA (indep.) column)
# Only requires DEVICE, RESULTS_DIR, SEED_LIST from Cell 2 (Stage 1). No retraining needed.
# =====================================================================================

import os, re, glob, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

METRICS    = ['STA', 'STA (indep.)', 'Sim', 'Flu', 'Joint']
CI_METRICS = METRICS

# -------------------------------------------------------------------------------------
# STEP 1 — Independent STA Scoring on pre-generated Y (inference only, no training)
# -------------------------------------------------------------------------------------
indep_tokenizer = AutoTokenizer.from_pretrained('s-nlp/roberta_toxicity_classifier')
indep_model = AutoModelForSequenceClassification.from_pretrained('s-nlp/roberta_toxicity_classifier').to(DEVICE)
indep_model.eval()

id2label_indep = indep_model.config.id2label
TOX_IDX_INDEP = next((int(i) for i, l in id2label_indep.items() if str(l).lower() in {'toxic', 'toxicity'}), 1)

@torch.no_grad()
def compute_sta_indep(texts, batch_size=32):
    out = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) for t in texts[i:i + batch_size]]
        enc = indep_tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to(DEVICE)
        probs = torch.softmax(indep_model(**enc).logits.float(), dim=-1)[:, TOX_IDX_INDEP]
        out.extend((1.0 - probs).cpu().tolist())
    return out

csv_files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'per_sample_seed*.csv'))) \
    + sorted(glob.glob(os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')))

print('=' * 90); print('STEP 1 — Independent STA Scoring'); print('=' * 90)
gap_rows = []
for path in csv_files:
    df = pd.read_csv(path)
    if 'STA (indep.)' not in df.columns:
        df['STA (indep.)'] = compute_sta_indep(df['Y'].tolist())
        df.to_csv(path, index=False)

    fname = os.path.basename(path)
    m = re.search(r'seed(\d+)_(.+)\.csv', fname)
    seed, phase = (m.group(1), m.group(2)) if m else ('baseline', 'baseline')
    gap_rows.append({'seed': seed, 'phase': phase, 'n': len(df),
                      'STA (orig.)': round(df['STA'].mean(), 4),
                      'STA (indep.)': round(df['STA (indep.)'].mean(), 4),
                      'gap': round(df['STA'].mean() - df['STA (indep.)'].mean(), 4)})
    print(f"{fname}: STA(orig)={df['STA'].mean():.3f} | STA(indep.)={df['STA (indep.)'].mean():.3f}")

sta_gap_table = pd.DataFrame(gap_rows).sort_values(['phase', 'seed'])
sta_gap_table.to_csv(os.path.join(RESULTS_DIR, 'sta_independent_comparison.csv'), index=False)
print(sta_gap_table.to_string(index=False))

del indep_model
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

# -------------------------------------------------------------------------------------
# STEP 2 — Rebuild table3/3b/3c + table4/4b/4c (aggregated from per_sample, no re-generation)
# -------------------------------------------------------------------------------------
print('\n' + '=' * 90); print('STEP 2 — Rebuild table3/table4'); print('=' * 90)

def summarize2(df):
    return {m: float(df[m].mean()) for m in METRICS if m in df.columns}

BASELINE_CSV = os.path.join(RESULTS_DIR, 'baseline_per_sample.csv')
baseline_per_sample = pd.read_csv(BASELINE_CSV)
baseline_metrics_full     = summarize2(baseline_per_sample)
baseline_metrics_toxic    = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'toxic'])
baseline_metrics_nontoxic = summarize2(baseline_per_sample[baseline_per_sample['label'] == 'nontoxic'])

for SEED in SEED_LIST:
    phase_files = {
        'Phase 1 (Easy)':    f'per_sample_seed{SEED}_phase1_easy.csv',
        'Phase 2 (+Med)':    f'per_sample_seed{SEED}_phase2_balance.csv',
        'Phase 3 (Full)':    f'per_sample_seed{SEED}_phase3_refinement.csv',
        'Phase 3+Post-Val.': f'per_sample_seed{SEED}_phase3_postval.csv',
    }
    t3, t3b, t3c = [{'Phase': 'Baseline (SFT)', **baseline_metrics_full}], \
                   [{'Phase': 'Baseline (SFT)', **baseline_metrics_toxic}], \
                   [{'Phase': 'Baseline (SFT)', **baseline_metrics_nontoxic}]

    phase3_df_this_seed = None
    for name, fname in phase_files.items():
        fpath = os.path.join(RESULTS_DIR, fname)
        if not os.path.exists(fpath):
            print(f'[Seed {SEED}] Missing {fpath} — skipping.'); continue
        df = pd.read_csv(fpath)
        if name == 'Phase 3 (Full)':
            phase3_df_this_seed = df
        t3.append({'Phase': name, **summarize2(df)})
        t3b.append({'Phase': name, **summarize2(df[df['label'] == 'toxic'])})
        t3c.append({'Phase': name, **summarize2(df[df['label'] == 'nontoxic'])})

    pd.DataFrame(t3).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3_seed{SEED}.csv'), index=False)
    pd.DataFrame(t3b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3b_seed{SEED}.csv'), index=False)
    pd.DataFrame(t3c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table3c_seed{SEED}.csv'), index=False)

    if phase3_df_this_seed is not None:
        t4, t4b, t4c = [], [], []
        for band in ['easy', 'medium', 'hard']:
            bf = phase3_df_this_seed[phase3_df_this_seed['band'] == band]
            bt = phase3_df_this_seed[(phase3_df_this_seed['band'] == band) & (phase3_df_this_seed['label'] == 'toxic')]
            bn = phase3_df_this_seed[(phase3_df_this_seed['band'] == band) & (phase3_df_this_seed['label'] == 'nontoxic')]
            if len(bf) > 0: t4.append({'Difficulty': band.capitalize(), 'Count': len(bf), **summarize2(bf)})
            if len(bt) > 0: t4b.append({'Difficulty': band.capitalize(), 'Count': len(bt), **summarize2(bt)})
            if len(bn) > 0: t4c.append({'Difficulty': band.capitalize(), 'Count': len(bn), **summarize2(bn)})
        t4.append({'Difficulty': 'Overall', 'Count': len(phase3_df_this_seed), **summarize2(phase3_df_this_seed)})
        t4b.append({'Difficulty': 'Overall', 'Count': len(phase3_df_this_seed[phase3_df_this_seed['label'] == 'toxic']),
                     **summarize2(phase3_df_this_seed[phase3_df_this_seed['label'] == 'toxic'])})
        t4c.append({'Difficulty': 'Overall', 'Count': len(phase3_df_this_seed[phase3_df_this_seed['label'] == 'nontoxic']),
                     **summarize2(phase3_df_this_seed[phase3_df_this_seed['label'] == 'nontoxic'])})
        pd.DataFrame(t4).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4_seed{SEED}.csv'), index=False)
        pd.DataFrame(t4b).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4b_seed{SEED}.csv'), index=False)
        pd.DataFrame(t4c).round(4).to_csv(os.path.join(RESULTS_DIR, f'table4c_seed{SEED}.csv'), index=False)

    print(f'[Seed {SEED}] Rebuild xong.')

def print_main_table2(prefix, index_col='Phase'):
    dfs = {}
    for SEED in SEED_LIST:
        path = os.path.join(RESULTS_DIR, f'{prefix}_seed{SEED}.csv')
        if os.path.exists(path):
            dfs[SEED] = pd.read_csv(path).set_index(index_col)
    if not dfs:
        print(f'No {prefix}_seed*.csv files found.'); return
    keys = dfs[SEED_LIST[0]].index.tolist()
    for k in keys:
        print(f'--- {k} ---')
        for metric in METRICS:
            values, used_seeds = [], []
            for SEED in SEED_LIST:
                if SEED in dfs and k in dfs[SEED].index and metric in dfs[SEED].columns:
                    values.append(dfs[SEED].loc[k, metric]); used_seeds.append(SEED)
            if not values: continue
            seed_str = ' | '.join(f'S{s}={v:.3f}' for s, v in zip(used_seeds, values))
            mean, std = np.mean(values), (np.std(values, ddof=1) if len(values) > 1 else 0.0)
            print(f'  {metric:14s}: {seed_str}  ->  Mean±Std = {mean:.4f} ± {std:.4f}')
    print()

for prefix, idx_col in [('table3', 'Phase'), ('table3b', 'Phase'), ('table3c', 'Phase'),
                          ('table4', 'Difficulty'), ('table4b', 'Difficulty'), ('table4c', 'Difficulty')]:
    print('=' * 90); print(f'MAIN TABLE — {prefix} (with STA (indep.))'); print('=' * 90)
    print_main_table2(prefix, idx_col)

# -------------------------------------------------------------------------------------
# STEP 3 — Rebuild Bootstrap CI (FULL/TOXIC/NON-TOXIC), add STA (indep.)
# -------------------------------------------------------------------------------------
print('\n' + '=' * 90); print('STEP 3 — Rebuild Bootstrap CI'); print('=' * 90)

def bootstrap_ci(values, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.default_rng(seed)
    values = np.asarray(values); n = len(values)
    boot_means = np.array([rng.choice(values, size=n, replace=True).mean() for _ in range(n_boot)])
    lo = np.percentile(boot_means, (1 - ci) / 2 * 100)
    hi = np.percentile(boot_means, (1 + ci) / 2 * 100)
    return values.mean(), lo, hi

_all = []
for SEED in SEED_LIST:
    for fname, phase_name in [(f'per_sample_seed{SEED}_phase1_easy.csv', 'Phase 1 (Easy)'),
                               (f'per_sample_seed{SEED}_phase2_balance.csv', 'Phase 2 (+Med)'),
                               (f'per_sample_seed{SEED}_phase3_refinement.csv', 'Phase 3 (Full)'),
                               (f'per_sample_seed{SEED}_phase3_postval.csv', 'Phase 3+Post-Val.')]:
        fpath = os.path.join(RESULTS_DIR, fname)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            df['seed'], df['phase'] = SEED, phase_name
            _all.append(df)
all_per_sample_pooled = pd.concat(_all, ignore_index=True)
print(f'Total per-sample lines reloaded: {len(all_per_sample_pooled)} rows')

def build_ci_table_pooled2(df, group_col='phase', label_filter=None):
    sub = df if label_filter is None else df[df['label'] == label_filter]
    rows = []
    for g in sub[group_col].unique():
        gdf = sub[sub[group_col] == g]
        row = {group_col: g, 'n_pooled': len(gdf)}
        for col in CI_METRICS:
            if col not in gdf.columns: continue
            mean, lo, hi = bootstrap_ci(gdf[col].values)
            row[col] = round(mean, 4); row[f'{col}_CI95'] = f'[{lo:.3f}, {hi:.3f}]'
        rows.append(row)
    return pd.DataFrame(rows)

for label, filt in [('FULL', None), ('TOXIC', 'toxic'), ('NON-TOXIC', 'nontoxic')]:
    ci_table = build_ci_table_pooled2(all_per_sample_pooled, label_filter=filt)
    print(f'\n=== Bootstrap CI (Appendix) — {label} (with STA (indep.)) ===')
    print(ci_table.to_string(index=False))
    ci_table.to_csv(os.path.join(RESULTS_DIR, f'bootstrap_ci_{label.lower().replace("-", "")}.csv'), index=False)

print('\nCOMPLETED — all tables (table3/3b/3c, table4/4b/4c, bootstrap_ci_*) now have STA (indep.) column.')